# Redistribute blobs

This notebook redistributes objects in an image within an "allowed area". Objects must be given as a *label image* i.e. an image in which each pixel value represents a different object. The allowed area must be given as a binary mask.

Start by uploading the label images and binary masks to separate Google Drive folders. Make sure these folders contain no other files beside the images. Create an empty folder to save randomized images.

In [ ]:
#@title Link Google Colaboratory to Drive account to access files

from google.colab import drive
drive.mount('/content/drive')

file_root = '/content/drive/My Drive/'

In [ ]:
#@title Define function to randomize object positions

import numpy as np
from skimage.measure import regionprops
from scipy.ndimage import binary_dilation


def randomize_labels(
    label_image,
    allowed_mask,
    n_randomizations=10,
    max_attempts=1000,
    max_total_attempts=None
):
    """
    Randomly redistributes labeled objects within an allowed area.

    Parameters
    ----------
    label_image : ndarray
        Label image. Background = 0, each object has a unique label.

    allowed_mask : ndarray of bool
        Binary mask defining where objects may be placed.

    n_randomizations : int
        Number of valid randomizations to generate.

    max_attempts : int
        Maximum placement attempts per object.

    max_total_attempts : int or None
        Maximum total randomization attempts. If None,
        defaults to 20 × n_randomizations.

    Returns
    -------
    list of ndarray
        List of successfully randomized label images.
    """

    if max_total_attempts is None:
        max_total_attempts = n_randomizations * 20

    # ------------------------------------------------------------------
    # Extract objects
    # ------------------------------------------------------------------

    objects = []

    for prop in regionprops(label_image):

        label = prop.label

        minr, minc, maxr, maxc = prop.bbox

        obj_mask = (
            label_image[minr:maxr, minc:maxc] == label
        )

        objects.append(
            {
                "label": label,
                "mask": obj_mask,
                "height": obj_mask.shape[0],
                "width": obj_mask.shape[1],
                "area": np.sum(obj_mask)
            }
        )

    # Place large objects first
    objects.sort(
        key=lambda x: x["area"],
        reverse=True
    )

    # ------------------------------------------------------------------
    # Generate randomizations
    # ------------------------------------------------------------------

    valid_randomizations = []
    total_attempts = 0

    while len(valid_randomizations) < n_randomizations:

        total_attempts += 1

        if total_attempts > max_total_attempts:
            print(
                f"Stopped after {total_attempts-1} attempts. "
                f"Generated {len(valid_randomizations)} "
                f"valid randomizations."
            )
            break

        output = np.zeros_like(label_image)

        # Randomize placement order each time
        shuffled_indices = np.random.permutation(len(objects))

        success = True

        for idx in shuffled_indices:

            obj = objects[idx]

            obj_mask = obj["mask"]
            h = obj["height"]
            w = obj["width"]
            label = obj["label"]

            placed = False

            for _ in range(max_attempts):

                row = np.random.randint(
                    0,
                    label_image.shape[0] - h + 1
                )

                col = np.random.randint(
                    0,
                    label_image.shape[1] - w + 1
                )

                allowed_crop = allowed_mask[
                    row:row+h,
                    col:col+w
                ]

                # Entire object must fit inside mask
                if not np.all(allowed_crop[obj_mask]):
                    continue

                output_crop = output[
                    row:row+h,
                    col:col+w
                ]

                # Prevent overlap
                if np.any(output_crop[obj_mask] > 0):
                    continue

                output_crop[obj_mask] = label
                placed = True
                break

            if not placed:
                success = False
                break

        if success:
            valid_randomizations.append(output)

            print(
                f"Generated "
                f"{len(valid_randomizations)}/"
                f"{n_randomizations}"
            )

    return valid_randomizations

In [ ]:
#@title Indicate folder paths
label_dir = "/content/drive/MyDrive/MIE/Redistribute MLOs/Label images/" # @param {"type":"string"}
mask_dir = "/content/drive/MyDrive/MIE/Redistribute MLOs/Cytoplasm masks/" # @param {"type":"string"}
output_dir = "/content/drive/MyDrive/MIE/Redistribute MLOs/Randomized MLOs/" # @param {"type":"string"}

In [ ]:
#@title Indicate number of randomizations per image, then run the code chunk. Randomized images will be saved in output folder.

from pathlib import Path
from skimage import io

# Number of randomizations per image
n_randomizations = 10 # @param {"type":"integer"}

for label_file in label_dir.glob("*.tif"):

    # Find matching mask
    mask_file = mask_dir / label_file.name

    if not mask_file.exists():
        print(f"No mask found for {label_file.name}, skipping.")
        continue

    print(f"Processing {label_file.name}")

    # Load image and mask
    label_img = io.imread(label_file)
    allowed_mask = io.imread(mask_file) > 0

    # Generate randomizations
    randomized_images = randomize_labels(
        label_img,
        allowed_mask,
        n_randomizations=n_randomizations
    )

    # Create output folder named after image
    image_output_dir = output_dir / label_file.stem
    image_output_dir.mkdir(exist_ok=True)

    # Save as 001.tif, 002.tif, ...
    for i, rnd_img in enumerate(randomized_images, start=1):

        out_path = image_output_dir / f"{i:03d}.tif"

        io.imsave(
            out_path,
            rnd_img.astype(label_img.dtype),
            check_contrast=False
        )

    print(f"Saved {len(randomized_images)} randomizations")

Processing C1-Control_vA_01_0.tif
Generated 1/10
Generated 2/10
Generated 3/10
Generated 4/10
Generated 5/10
Generated 6/10
Generated 7/10
Generated 8/10
Generated 9/10
Generated 10/10
Saved 10 randomizations
Processing C1-Control_vA_01_1.tif
Generated 1/10
Generated 2/10
Generated 3/10
Generated 4/10
Generated 5/10
Generated 6/10
Generated 7/10
Generated 8/10
Generated 9/10
Generated 10/10
Saved 10 randomizations
Processing C1-Control_vA_02_0.tif
Generated 1/10
Generated 2/10
Generated 3/10
Generated 4/10
Generated 5/10
Generated 6/10
Generated 7/10
Generated 8/10
Generated 9/10
Generated 10/10
Saved 10 randomizations
Processing C1-Control_vA_03_0.tif
Generated 1/10
Generated 2/10
Generated 3/10
Generated 4/10
Generated 5/10
Generated 6/10
Generated 7/10
Generated 8/10
Generated 9/10
Generated 10/10
Saved 10 randomizations
Processing C1-Control_vA_05_0.tif
Generated 1/10
Generated 2/10
Generated 3/10
Generated 4/10
Generated 5/10
Generated 6/10
Generated 7/10
Generated 8/10
Generated 